# Eksperyment: Uczciwa walidacja wariantów slice-aware (walk-forward + McNemar)

## Cel
Warianty **slice-aware** (`sliceaware`, `bestof5_v1`, `qfserve_v3`) były dotąd oceniane
**tylko na pojedynczym teście** (sezon 2024/2025). Tam dawały spektakularne wyniki:
`bestof5_v1` raportował **+2.37 p.p.**, `qfserve_v3` **+2.20 p.p.** match accuracy nad baseline.
Ale jeden sezon potrafi skłamać. Tu sprawdzamy te warianty **uczciwie**.

## Metoda (walk-forward + test parowany)
- **Walk-forward** przez 6 sezonów (2020-2025): dla *każdego* roku osobno trenujemy baseline
  i każdy wariant na **identycznych** meczach (te same splity, ten sam seed).
- Baseline liczy się **raz na rok** i jest cache'owany (monkey-patch `runpy.run_path`), a 3 warianty
  go reużywają -- inaczej 4x pełny baseline na rok byłoby marnotrawstwem.
- **Test parowany McNemar**: porównujemy mecz-po-meczu (po `match_id`), kto trafił baseline vs wariant.
  Liczymy `b` = baseline trafił & wariant nie, `c` = wariant trafił & baseline nie. McNemar patrzy
  tylko na te *niezgodne* pary -- to właściwy test, bo pojedyncza pooled delta nie mówi nic o istotności.
- Pooled po wszystkich latach (N~3022 meczów) -> jeden wynik per wariant: pooled delta + p-value.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

## 1. Reuse modułu walidacji (bez `main()`)
Importujemy gotowe prymitywy z `tennis_model_validate_variants.py` zamiast duplikować logikę:

- `execute_script(name)` -- uruchamia skrypt przez (zmonkeypatchowane) `runpy.run_path` i zwraca jego
  namespace; baseline jest cache'owany per rok.
- `eval_frame(ns)` -- wyciąga z namespace'u `winner_perspective[["match_id", "correct_prediction"]]`
  (jeden wiersz na mecz: czy model trafił zwycięzcę).
- `mcnemar(b, c)` -- statystyka z poprawką ciągłości + dwustronne p (`erfc`).
- `reset_baseline_cache()` -- czyści cache baseline na początku każdego roku.
- `TARGET_YEARS`, `VARIANTS` -- definicja walk-forward i mapowanie wariant->skrypt.

**Uwaga o leakage/spójności:** każdy skrypt sam robi chronologiczny split 60/20/20 i `RANDOM_STATE=42`;
warianty importują namespace baseline (te same mecze, te same tuned HP), więc porównanie jest czystą
ablacją -- zmieniamy tylko cechy.

In [2]:
import numpy as np
import pandas as pd

import tennis_model_validate_variants as m
from tennis_model_validate_variants import (
    execute_script, eval_frame, mcnemar, reset_baseline_cache,
    TARGET_YEARS, VARIANTS,
)

print(f"Walk-forward lata: {TARGET_YEARS}")
print("Warianty pod testem (wariant -> skrypt):")
for name, script in VARIANTS.items():
    print(f"  {name:<12} <- {script}")
print(f"\nBaseline (wspólny rdzeń): tennis_model.py  |  metryka: match_accuracy")

Walk-forward lata: [2020, 2021, 2022, 2023, 2024, 2025]
Warianty pod testem (wariant -> skrypt):
  sliceaware   <- tennis_model_sliceaware.py
  bestof5_v1   <- tennis_model_sliceaware_bestof5_v1.py
  qfserve_v3   <- tennis_model_sliceaware_qfserve_v3.py

Baseline (wspólny rdzeń): tennis_model.py  |  metryka: match_accuracy


## 2. Sanity-check McNemara
Zanim ruszymy ciężki bieg, sprawdźmy intuicję testu na ręcznych liczbach. McNemar patrzy WYŁĄCZNIE na
niezgodne pary (`b` vs `c`); zgodne (oba trafiły / oba spudłowały) nie niosą informacji o różnicy.
- duża, jednostronna przewaga (`b`=10, `c`=40) -> małe p (istotne),
- symetryczna niezgoda (`b`=25, `c`=25) -> p~1 (brak różnicy),
- brak niezgodnych par (`b`=`c`=0) -> p=1 z definicji.

In [3]:
for b, c in [(10, 40), (25, 25), (0, 0), (40, 10)]:
    z, p = mcnemar(b, c)
    verdict = "ISTOTNE" if p < 0.05 else "brak istotności"
    direction = "na korzyść wariantu" if c > b else ("na niekorzyść wariantu" if b > c else "remis")
    print(f"  b={b:>2} c={c:>2}  ->  z={z:4.2f}  p={p:.4f}  [{verdict}, {direction}]")

  b=10 c=40  ->  z=4.10  p=0.0000  [ISTOTNE, na korzyść wariantu]
  b=25 c=25  ->  z=-0.14  p=0.8875  [brak istotności, remis]
  b= 0 c= 0  ->  z=0.00  p=1.0000  [brak istotności, remis]
  b=40 c=10  ->  z=4.10  p=0.0000  [ISTOTNE, na niekorzyść wariantu]


## 3. Demo: jeden sezon (2025)
Najpierw narracyjnie rozbijmy *jeden* rok, żeby pokazać dokładnie, co robi pętla. Ustawiamy rok przez
`TENNIS_TARGET_YEAR`, czyścimy cache baseline, liczymy baseline **raz**, potem każdy wariant reużywa
cache'owany baseline. Parujemy `winner_perspective` po `match_id` i pokazujemy macierz zgodności
(`b`/`c`) dla pojedynczego wariantu -- to ten sam mechanizm, który zaraz zapętlimy po latach.

To **jednorazowy, dawny rodzaj testu** -- i właśnie tu rodziły się spektakularne pojedyncze wyniki.

In [4]:
import os

demo_year = 2025
os.environ["TENNIS_TARGET_YEAR"] = str(demo_year)
reset_baseline_cache()

# baseline raz (zapelnia cache); warianty ponizej go reuzyja
base_ns = execute_script("tennis_model.py")
base_eval = eval_frame(base_ns)
base_match = float(base_ns["match_accuracy"])
print(f"[{demo_year}] baseline match={base_match:.4f}  (n={len(base_eval)} meczów)")

# przyklad parowania dla jednego wariantu
demo_name = "bestof5_v1"
var_ns = execute_script(VARIANTS[demo_name])      # reuzywa cached baseline
var_eval = eval_frame(var_ns)
var_match = float(var_ns["match_accuracy"])
merged = base_eval.merge(var_eval, on="match_id", suffixes=("_base", "_var"))

bc = merged["correct_prediction_base"].astype(bool)
vc = merged["correct_prediction_var"].astype(bool)
b = int((bc & ~vc).sum()); c = int((~bc & vc).sum())
both = int((bc & vc).sum()); neither = int((~bc & ~vc).sum())
z, p = mcnemar(b, c)
print(f"[{demo_year}] {demo_name} match={var_match:.4f}  single-test delta={var_match-base_match:+.4f}")
print(f"   macierz zgodności: oba_OK={both}  oba_MISS={neither}  tylko_baseline(b)={b}  tylko_wariant(c)={c}")
print(f"   McNemar (1 sezon): z={z:.2f} p={p:.4f}  -> jeden rok to za mało, by cokolwiek orzec")

[2025] baseline match=0.6566  (n=530 meczów)


[2025] bestof5_v1 match=0.6679  single-test delta=+0.0113
   macierz zgodności: oba_OK=328  oba_MISS=156  tylko_baseline(b)=20  tylko_wariant(c)=26
   McNemar (1 sezon): z=0.74 p=0.4610  -> jeden rok to za mało, by cokolwiek orzec


## 4. Walk-forward: widoczna pętla po sezonach 2020-2025
Teraz to samo dla **każdego** roku i **każdego** wariantu, akumulując pary (base_correct, var_correct)
per wariant w `pairs` oraz wyniki per rok w `per_year`. Struktura identyczna jak w module
(`pairs`, `per_year`), ale rozpisana na widoku, z drukowaniem postępów.

To długi bieg: 6 lat x (1 baseline + 3 warianty), każde to pełny trening RF. Cache sprawia, że baseline
liczy się raz na rok.

In [5]:
pairs = {name: [] for name in VARIANTS}      # (base_correct, var_correct)
per_year = {name: [] for name in VARIANTS}

for year in TARGET_YEARS:
    print(f"\n===== ROK {year} =====", flush=True)
    os.environ["TENNIS_TARGET_YEAR"] = str(year)
    reset_baseline_cache()

    # baseline raz na rok (zapelnia cache, ktory reuzyja warianty)
    base_ns = execute_script("tennis_model.py")
    base_eval = eval_frame(base_ns)
    base_match = float(base_ns["match_accuracy"])
    print(f"  baseline match={base_match:.4f}  (n={len(base_eval)})", flush=True)

    for name, script in VARIANTS.items():
        var_ns = execute_script(script)          # reuzywa cached baseline
        var_eval = eval_frame(var_ns)
        var_match = float(var_ns["match_accuracy"])
        merged = base_eval.merge(var_eval, on="match_id", suffixes=("_base", "_var"))
        for _, r in merged.iterrows():
            pairs[name].append((bool(r["correct_prediction_base"]), bool(r["correct_prediction_var"])))
        per_year[name].append({"year": year, "baseline": base_match, "variant": var_match,
                               "delta": var_match - base_match})
        print(f"    {name:<12} match={var_match:.4f}  delta={var_match-base_match:+.4f}", flush=True)

os.environ.pop("TENNIS_TARGET_YEAR", None)
print("\nWalk-forward zakończony.")


===== ROK 2020 =====


  baseline match=0.6176  (n=272)


    sliceaware   match=0.5919  delta=-0.0257


    bestof5_v1   match=0.6213  delta=+0.0037


    qfserve_v3   match=0.6066  delta=-0.0110


...

  baseline match=0.6566  (n=530)


    sliceaware   match=0.6528  delta=-0.0038


    bestof5_v1   match=0.6679  delta=+0.0113


    qfserve_v3   match=0.6358  delta=-0.0208



Walk-forward zakończony.


## 5. Per rok: tabele delt
Dla każdego wariantu pokazujemy tabelę rok-po-roku (baseline vs wariant vs delta) oraz w ilu sezonach
delta była dodatnia. Tu zwykle widać, że pojedynczy "spektakularny" sezon jest wyjątkiem, a nie
regułą -- delty skaczą w obie strony.

In [6]:
for name in VARIANTS:
    df = pd.DataFrame(per_year[name])
    pos = int((df["delta"] > 0).sum())
    print(f"--- {name} ---")
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"  delta dodatnia w {pos}/{len(df)} sezonach\n")

--- sliceaware ---
 year  baseline  variant   delta
 2020    0.6176   0.5919 -0.0257
 2021    0.6667   0.6762  0.0096
 2022    0.6728   0.6782  0.0055
 2023    0.6346   0.6239 -0.0107
 2024    0.6186   0.6169 -0.0017
 2025    0.6566   0.6528 -0.0038
  delta dodatnia w 2/6 sezonach

--- bestof5_v1 ---
 year  baseline  variant   delta
 2020    0.6176   0.6213  0.0037
 2021    0.6667   0.6705  0.0038
 2022    0.6728   0.6709 -0.0018
 2023    0.6346   0.6364  0.0018
 2024    0.6186   0.6322  0.0136
 2025    0.6566   0.6679  0.0113
  delta dodatnia w 5/6 sezonach

--- qfserve_v3 ---
 year  baseline  variant   delta
 2020    0.6176   0.6066 -0.0110
 2021    0.6667   0.6533 -0.0134
 2022    0.6728   0.6746  0.0018
 2023    0.6346   0.6168 -0.0178
 2024    0.6186   0.6237  0.0051
 2025    0.6566   0.6358 -0.0208
  delta dodatnia w 2/6 sezonach



## 6. Pooled + McNemar (werdykt)
Łączymy wszystkie mecze ze wszystkich lat i liczymy per wariant: pooled accuracy baseline vs wariant,
pooled delta oraz McNemar (`b`, `c`, `z`, `p`). To jest *właściwa* odpowiedź na pytanie "czy wariant
istotnie pobija główny model" -- pojedynczy sezon zostaje rozcieńczony w N~3022 meczach.

In [7]:
print("=" * 74)
print("WALK-FORWARD: warianty slice-aware vs baseline  (pooled 2020-2025)")
print("=" * 74)
for name in VARIANTS:
    df = pd.DataFrame(per_year[name])
    arr = np.array(pairs[name])
    base_c, var_c = arr[:, 0], arr[:, 1]
    N = len(arr)
    b = int(np.sum(base_c & ~var_c))
    c = int(np.sum(~base_c & var_c))
    z, p = mcnemar(b, c)
    pooled_delta = var_c.mean() - base_c.mean()
    pos = int((df["delta"] > 0).sum())
    if p < 0.05 and c > b:
        verdict = "ISTOTNE na korzyść wariantu"
    elif p < 0.05 and b > c:
        verdict = "ISTOTNE na niekorzyść wariantu"
    else:
        verdict = "brak istotności (p>=0.05)"
    print(f"\n--- {name} ---")
    print(f"  POOLED ({N} meczów): baseline={base_c.mean():.4f}  {name}={var_c.mean():.4f}  "
          f"delta={pooled_delta:+.4f}  (dodatnie sezony {pos}/{len(df)})")
    print(f"  McNemar: b={b} c={c} z={z:.2f} p={p:.4f}  -> {verdict}")

WALK-FORWARD: warianty slice-aware vs baseline  (pooled 2020-2025)

--- sliceaware ---
  POOLED (3022 meczów): baseline=0.6463  sliceaware=0.6436  delta=-0.0026  (dodatnie sezony 2/6)
  McNemar: b=98 c=90 z=0.51 p=0.6097  -> brak istotności (p>=0.05)

--- bestof5_v1 ---
  POOLED (3022 meczów): baseline=0.6463  bestof5_v1=0.6519  delta=+0.0056  (dodatnie sezony 5/6)
  McNemar: b=114 c=131 z=1.02 p=0.3067  -> brak istotności (p>=0.05)

--- qfserve_v3 ---
  POOLED (3022 meczów): baseline=0.6463  qfserve_v3=0.6373  delta=-0.0089  (dodatnie sezony 2/6)
  McNemar: b=121 c=94 z=1.77 p=0.0762  -> brak istotności (p>=0.05)


## Wnioski
Walidacja przez 6 sezonów (2020–2025, ~3000 meczów, baseline 0,6463) rozwiała optymizm z pojedynczych testów:

| wariant | zmiana | McNemar p |
|---|---|---|
| sliceaware | −0,26 p.p. | 0,61 |
| bestof5_v1 | +0,56 p.p. | 0,31 |
| qfserve_v3 | −0,89 p.p. | 0,076 |

Żaden wariant nie jest istotny (p > 0,05). Dawne efektowne wyniki z jednego sezonu (bestof5 +2,37 p.p., qfserve +2,20 p.p.) rozpłynęły się w szumie, gdy zmierzyłem je uczciwie przez 6 sezonów: bestof5 jest minimalnie dodatni, ale daleki od istotności; qfserve wręcz kierunkowo gorszy; sliceaware neutralny. To dokładnie lekcja całego projektu — pojedynczy test potrafi skłamać o jakieś 2 p.p., a prawdę mówi dopiero walidacja przez wiele sezonów z testem parowanym.